# TypeDict的使用

In [1]:

from scripts.regsetup import description
from typing_extensions import TypedDict
from rich import print as rprint


class TestDict(TypedDict):
    id: int
    name: str
    age: int
    address: str


movie: TestDict = {
    'id': 1,
    'name': "张三",
    'age': 21,
    'address': "张三省的",
}
rprint(movie)

{'id': 1, 'name': '张三', 'age': 21, 'address': '张三省的'}

# 元数据 Annotated 类似于Field

In [ ]:
from anthropic import BaseModel
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from typing_extensions import TypedDict, Annotated

from pydantic import Field

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-chat",
    # api_key = DEEPSEEK_API_KEY,
    # url = DEEPSEEK_API_URL
    extra_body={
        "thinking": {
            "type": "disabled"  # 🔑 关键：关闭思考模式
        }
    }
)

"""
使用 TypedDict 模型定义结构化输出
"""
class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """

    title: Annotated[str, "电影的正式名称，例如《盗梦空间》,包括它的简要概括"]
    year: Annotated[int, "电影的公映年份，使用四位数字表示"]
    director: Annotated[str, "电影导演的全名"]
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]
    acters: Annotated[str, "电影的主角以及演员名字"]
# 设置模型结构化输出
structured_llm = model.with_structured_output(MovieTypedDict)
# 调用模型并获取结构化输出
response = structured_llm.invoke(
    "请提供电影《这个杀手不太冷》的完整信息，"
    "必须包含：title（标题,简要概括，有什么意义）、year（上映年份）、director（导演）、rating（豆瓣评分，满分10分）。acters（演员,最少说3个，）"
    "请确保返回所有四个字段。"
)
rprint(type(response))
rprint(response)


## @dataClass 测试
结构化输出

In [13]:

from langchain.chat_models import init_chat_model
from dataclasses import dataclass, field
from rich import print as rprint
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-chat",
    # api_key = DEEPSEEK_API_KEY,
    # url = DEEPSEEK_API_URL
    extra_body={
        "thinking": {
            "type": "disabled"  # 🔑 关键：关闭思考模式
        }
    }
)
@dataclass
class FlowersDataClass():
    """
    这是详细描述一朵花的信息
    """


    showing : str
    country: str
    favorite: str
    name: str = field(default="丁香花")
    color : str = field(default="white")
# flowers = FlowersDataClass( country="中国", color="blue", showing="green")
# rprint(flowers)



structured_llm = model.with_structured_output(FlowersDataClass,include_raw=False)
response = structured_llm.invoke("你是一个花朵研究者,用FlowersDataClass函数说明玫瑰花的一些特性，比如形态、颜色、主要国家、以及名字，说出你的最优解(最喜欢的)")
rprint(type(response))  #  表示返回的是Python 字典（dict）类型。
rprint(response)

<class 'dict'>

{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 260,
                'prompt_tokens': 374,
                'total_tokens': 634,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                'prompt_cache_hit_tokens': 256,
                'prompt_cache_miss_tokens': 118
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
            'id': 'cd736993-e418-435e-b87a-fafa6d44be51',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f84ab-05de-7802-a5d2-3118a3e043a7-0',
        tool_calls=[
            {
                'name': 'FlowersDataClass',
                'args': {
                    'showing': 
'玫瑰花开时呈碗状或杯状，花瓣多层重叠，边缘微卷，形态优雅。花径通常5-13厘米，花瓣数量可多达40-100片。茎干有刺，叶片
为羽状复叶，边缘有锯齿。',
                    'country': '保加利亚（著名的大马士革玫瑰产地）、法国、土耳其、中国、伊朗、美国等',
                    'favorite': 
'最优解（最喜欢）：玫瑰花！它不仅是爱情的象征，更拥有丰富的品种多样性、迷人的芬芳和优雅的层叠花瓣。无论是红玫瑰的热
烈、白玫瑰的纯洁还是粉玫瑰的温柔，都令人沉醉。玫瑰既可观赏又可制精油、入茶入馔，堪称花中皇后。',
                    'name': '玫瑰花',
                    'color': '红、粉、白、黄、橙、紫、蓝（染色/培育）、绿等多种颜色'
                },
                'id': 'call_00_gwx9BafgMcliakm1dfV27221',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 374,
            'output_tokens': 260,
            'total_tokens': 634,
            'input_token_details': {'cache_read': 256},
            'output_token_details': {}
        }
    ),
    'parsed': {
        'showing': 
'玫瑰花开时呈碗状或杯状，花瓣多层重叠，边缘微卷，形态优雅。花径通常5-13厘米，花瓣数量可多达40-100片。茎干有刺，叶片
为羽状复叶，边缘有锯齿。',
        'country': '保加利亚（著名的大马士革玫瑰产地）、法国、土耳其、中国、伊朗、美国等',
        'favorite': 
'最优解（最喜欢）：玫瑰花！它不仅是爱情的象征，更拥有丰富的品种多样性、迷人的芬芳和优雅的层叠花瓣。无论是红玫瑰的热
烈、白玫瑰的纯洁还是粉玫瑰的温柔，都令人沉醉。玫瑰既可观赏又可制精油、入茶入馔，堪称花中皇后。',
        'name': '玫瑰花',
        'color': '红、粉、白、黄、橙、紫、蓝（染色/培育）、绿等多种颜色'
    },
    'parsing_error': None
}